## Setup

Import shared utilities from `src/utils` and set up the project path. This notebook builds the Silver layer: resolves team ids from Bronze
into canonical team_key values using teams_master.

In [1]:
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd()
sys.path.append(str(PROJECT_ROOT / "src"))

from utils.parquet_io import read_league_all_seasons, read_partition, write_partition
from utils.team_resolution import resolve_team_key
import pyarrow as pa
from utils.schema import MATCHES_SILVER_SCHEMA

## Load Bronze matches and teams_master

Load all seasons for both leagues from Bronze, plus the reference teams_master table used to resolve team_key.

In [2]:
LAKE_ROOT = PROJECT_ROOT / "data"

laliga_bronze = read_league_all_seasons(LAKE_ROOT, layer="bronze", league="laliga").to_pandas()
primeira_bronze = read_league_all_seasons(LAKE_ROOT, layer="bronze", league="primeira_liga").to_pandas()

teams_master = read_partition(LAKE_ROOT / "reference" / "teams_master.parquet").to_pandas()

print(f"LaLiga bronze: {len(laliga_bronze)} matches, seasons: {sorted(laliga_bronze['season'].unique())}")
print(f"Primeira Liga bronze: {len(primeira_bronze)} matches, seasons: {sorted(primeira_bronze['season'].unique())}")
print(f"teams_master: {len(teams_master)} teams")

LaLiga bronze: 1520 matches, seasons: ['2023-2024', '2024-2025', '2025-2026', '2026-2027']
Primeira Liga bronze: 1224 matches, seasons: ['2023-2024', '2024-2025', '2025-2026', '2026-2027']
teams_master: 54 teams


## Diagnostic: team id coverage

Check how many home/away team ids in Bronze fail to match teams_master.football_data_org_id, before calling resolve_team_key
(which raises on any unresolved id).

In [3]:
known_ids = set(teams_master["football_data_org_id"])

for name, df in [("LaLiga", laliga_bronze), ("Primeira Liga", primeira_bronze)]:
    unmatched_home = set(df["home_team_id"]) - known_ids
    unmatched_away = set(df["away_team_id"]) - known_ids
    unmatched = unmatched_home | unmatched_away
    print(f"{name}: {len(unmatched)} ids sin match -> {unmatched}")

LaLiga: 0 ids sin match -> set()
Primeira Liga: 0 ids sin match -> set()


In [4]:
for name, df in [("LaLiga", laliga_bronze), ("Primeira Liga", primeira_bronze)]:
    unmatched_ids = (set(df["home_team_id"]) | set(df["away_team_id"])) - known_ids
    mask = df["home_team_id"].isin(unmatched_ids) | df["away_team_id"].isin(unmatched_ids)
    subset = df.loc[mask, ["season", "home_team", "home_team_id", "away_team", "away_team_id"]]

    teams_seen = set(subset.loc[subset["home_team_id"].isin(unmatched_ids), "home_team"]) | \
                 set(subset.loc[subset["away_team_id"].isin(unmatched_ids), "away_team"])
    seasons_seen = sorted(subset["season"].unique())

    print(f"{name}: {teams_seen}")
    print(f"  temporadas afectadas: {seasons_seen}\n")

LaLiga: set()
  temporadas afectadas: []

Primeira Liga: set()
  temporadas afectadas: []



## Resolve team_key

Apply resolve_team_key to both leagues now that teams_master has full
historical coverage (54 teams).

In [5]:
laliga_silver = resolve_team_key(laliga_bronze, teams_master)
primeira_silver = resolve_team_key(primeira_bronze, teams_master)

print(laliga_silver[["home_team", "home_team_id", "home_team_key", "away_team", "away_team_id", "away_team_key"]].head())

                 home_team home_team_id            home_team_key  \
0               UD Almería          267               ud_almería   
1               Sevilla FC          559               sevilla_fc   
2  Real Sociedad de Fútbol           92  real_sociedad_de_fútbol   
3            UD Las Palmas          275            ud_las_palmas   
4            Athletic Club           77            athletic_club   

                  away_team away_team_id             away_team_key  
0  Rayo Vallecano de Madrid           87  rayo_vallecano_de_madrid  
1               Valencia CF           95               valencia_cf  
2                 Girona FC          298                 girona_fc  
3              RCD Mallorca           89              rcd_mallorca  
4            Real Madrid CF           86            real_madrid_cf  


## Compute result and goal_difference

Add the two derived columns from MATCHES_SILVER_SCHEMA. Only defined
for finished matches — home_score/away_score are null for matches not
yet played (status != FINISHED), so result/goal_difference stay null
too in those rows.

In [6]:
def add_result_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["goal_difference"] = df["home_score"] - df["away_score"]

    conditions = [
        df["home_score"] > df["away_score"],
        df["home_score"] < df["away_score"],
        df["home_score"] == df["away_score"],
    ]
    choices = ["H", "A", "D"]
    df["result"] = pd.Series(pd.NA, index=df.index, dtype="object")
    for cond, val in zip(conditions, choices):
        df.loc[cond & df["home_score"].notna() & df["away_score"].notna(), "result"] = val

    return df

laliga_silver = add_result_columns(laliga_silver)
primeira_silver = add_result_columns(primeira_silver)

print(laliga_silver["result"].value_counts(dropna=False))
print(laliga_silver[["home_score", "away_score", "goal_difference", "result"]].head())

result
H       553
A       342
D       315
<NA>    310
Name: count, dtype: int64
   home_score  away_score  goal_difference result
0         0.0         2.0             -2.0      A
1         1.0         2.0             -1.0      A
2         1.0         1.0              0.0      D
3         1.0         1.0              0.0      D
4         0.0         2.0             -2.0      A


In [7]:
laliga_silver["goal_difference"] = laliga_silver["goal_difference"].astype("Int32")
primeira_silver["goal_difference"] = primeira_silver["goal_difference"].astype("Int32")

In [8]:
SILVER_COLS = [f.name for f in MATCHES_SILVER_SCHEMA]

def write_silver(df: pd.DataFrame, league: str):
    for season, season_df in df.groupby("season"):
        table = pa.Table.from_pandas(season_df[SILVER_COLS], schema=MATCHES_SILVER_SCHEMA, preserve_index=False)
        path = LAKE_ROOT / "silver" / league / f"season={season}" / "matches.parquet"
        write_partition(table, path)
        print(f"{league} {season}: {len(season_df)} matches -> {path.relative_to(PROJECT_ROOT)}")

write_silver(laliga_silver, "laliga")
write_silver(primeira_silver, "primeira_liga")

laliga 2023-2024: 380 matches -> data\silver\laliga\season=2023-2024\matches.parquet
laliga 2024-2025: 380 matches -> data\silver\laliga\season=2024-2025\matches.parquet
laliga 2025-2026: 380 matches -> data\silver\laliga\season=2025-2026\matches.parquet
laliga 2026-2027: 380 matches -> data\silver\laliga\season=2026-2027\matches.parquet
primeira_liga 2023-2024: 306 matches -> data\silver\primeira_liga\season=2023-2024\matches.parquet
primeira_liga 2024-2025: 306 matches -> data\silver\primeira_liga\season=2024-2025\matches.parquet
primeira_liga 2025-2026: 306 matches -> data\silver\primeira_liga\season=2025-2026\matches.parquet
primeira_liga 2026-2027: 306 matches -> data\silver\primeira_liga\season=2026-2027\matches.parquet
